In [5]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.postgres import PostgresSaver
import os
from psycopg import Connection
from langchain_core.messages.utils import trim_messages,count_tokens_approximately
from langchain.messages import RemoveMessage, HumanMessage

In [2]:
load_dotenv()

True

In [3]:
llm = ChatGroq(model= "llama-3.1-8b-instant")

In [4]:
class ChatState(MessagesState):
    summary: str

In [60]:
def summarize(state: ChatState):
    summary = state.get("summary", "")

    if summary:
        prompt = (
            f"Existing summary:\n{summary}\n\n"
            "Extend the summary using the new conversation above."
        )
    else:
        prompt = "Summarize the conversation above."

    messages_for_summary = state['messages'] + [HumanMessage(content=prompt)]

    response = llm.invoke(messages_for_summary)

    messages_to_delete = state['messages'][:-2]

    return {
        'summary': response.content,
        'messages': [RemoveMessage(id= m.id) for m in messages_to_delete]
    }


In [61]:
def chat_node(state: ChatState):
    messages = []

    if state.get('summary'):
        messages.append({
            'role': 'system',
            "content": f"Conversation summary:\n{state['summary']}"
        })

    messages.extend(state['messages'])

    print(messages)

    response = llm.invoke(messages)

    return {'messages': [response]}

In [62]:
def should_summarize(state: ChatState):
    return len(state["messages"]) > 6

In [63]:
builder = StateGraph(MessagesState)

builder.add_node("chat", chat_node)
builder.add_node("summarize", summarize)

builder.add_edge(START, "chat")
builder.add_conditional_edges('chat', should_summarize, {
    True: 'summarize',
    False: END
})
builder.add_edge('summarize', END)


In [64]:
DB_URI = os.getenv("DATABASE_URL")

In [65]:
conn = Connection.connect(
    DB_URI,
    autocommit= True,
    prepare_threshold= None 
)

In [66]:
checkpointer = PostgresSaver(conn)
checkpointer.setup()

In [67]:
graph = builder.compile(checkpointer=checkpointer)

# Thread 2 (remembers)
t3 = {"configurable": {"thread_id": "thread-3"}}
graph.invoke({"messages": [{"role": "user", "content": "Hi, my name is Aryan"}]}, t3)

[HumanMessage(content='Hi, my name is Aryan', additional_kwargs={}, response_metadata={}, id='2def85a0-351c-4739-ab2a-4cba1a637f25'), AIMessage(content="Hello Aryan, it's nice to meet you. Is there something I can help you with or would you like to chat about a particular topic?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 42, 'total_tokens': 73, 'completion_time': 0.038691572, 'completion_tokens_details': None, 'prompt_time': 0.001977591, 'prompt_tokens_details': None, 'queue_time': 0.056209693, 'total_time': 0.040669163}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e64f9-35e5-72f0-801a-2d6dafedaab4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 31, 'total_tokens': 73}), HumanMessage(content='I am a SWE', additional_kwargs={}, response_me

{'messages': [HumanMessage(content='Hi, my name is Aryan', additional_kwargs={}, response_metadata={}, id='ea5eed27-0f65-4d2f-beb7-b1c6e0fb9c6a'),
  AIMessage(content="Aryan, it seems like you're repeating your name again. If you're ready, I can try to recall our previous conversation and we can continue from there. If not, we can start a new conversation. What would you prefer?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 568, 'total_tokens': 617, 'completion_time': 0.074412053, 'completion_tokens_details': None, 'prompt_time': 0.083610101, 'prompt_tokens_details': None, 'queue_time': 0.159431065, 'total_time': 0.158022154}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e6500-c7f4-77c2-ac78-09cc5224812c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 568, 'outpu

In [68]:
graph.invoke({"messages": [{"role": "user", "content": "I am a SWE"}]}, t3)

[{'role': 'system', 'content': 'Conversation summary:\nHere\'s a summary of our conversation:\n\n* You introduced yourself as Aryan and mentioned that you are a Software Engineer (SWE).\n* We discussed your background and interests in software engineering.\n* You mentioned that you live in Noida, a major tech hub in India.\n* You\'re pursuing a B. Tech degree, likely in Computer Science and Engineering (CSE).\n* We had a few conversations where you repeated your name, "Aryan", multiple times, and I tried to respond accordingly.'}, HumanMessage(content='Hi, my name is Aryan', additional_kwargs={}, response_metadata={}, id='ea5eed27-0f65-4d2f-beb7-b1c6e0fb9c6a'), AIMessage(content="Aryan, it seems like you're repeating your name again. If you're ready, I can try to recall our previous conversation and we can continue from there. If not, we can start a new conversation. What would you prefer?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_token

{'messages': [HumanMessage(content='Hi, my name is Aryan', additional_kwargs={}, response_metadata={}, id='ea5eed27-0f65-4d2f-beb7-b1c6e0fb9c6a'),
  AIMessage(content="Aryan, it seems like you're repeating your name again. If you're ready, I can try to recall our previous conversation and we can continue from there. If not, we can start a new conversation. What would you prefer?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 568, 'total_tokens': 617, 'completion_time': 0.074412053, 'completion_tokens_details': None, 'prompt_time': 0.083610101, 'prompt_tokens_details': None, 'queue_time': 0.159431065, 'total_time': 0.158022154}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e6500-c7f4-77c2-ac78-09cc5224812c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 568, 'outpu

In [69]:
graph.invoke({"messages": [{"role": "user", "content": "I live in Noida"}]}, t3)
graph.get_state(t3).values

[{'role': 'system', 'content': 'Conversation summary:\nHere\'s a summary of our conversation:\n\n* You introduced yourself as Aryan and mentioned that you are a Software Engineer (SWE).\n* We discussed your background and interests in software engineering.\n* You mentioned that you live in Noida, a major tech hub in India.\n* You\'re pursuing a B. Tech degree, likely in Computer Science and Engineering (CSE).\n* We had a few conversations where you repeated your name, "Aryan", multiple times, and I tried to respond accordingly.'}, HumanMessage(content='Hi, my name is Aryan', additional_kwargs={}, response_metadata={}, id='ea5eed27-0f65-4d2f-beb7-b1c6e0fb9c6a'), AIMessage(content="Aryan, it seems like you're repeating your name again. If you're ready, I can try to recall our previous conversation and we can continue from there. If not, we can start a new conversation. What would you prefer?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_token

{'messages': [HumanMessage(content='Hi, my name is Aryan', additional_kwargs={}, response_metadata={}, id='ea5eed27-0f65-4d2f-beb7-b1c6e0fb9c6a'),
  AIMessage(content="Aryan, it seems like you're repeating your name again. If you're ready, I can try to recall our previous conversation and we can continue from there. If not, we can start a new conversation. What would you prefer?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 568, 'total_tokens': 617, 'completion_time': 0.074412053, 'completion_tokens_details': None, 'prompt_time': 0.083610101, 'prompt_tokens_details': None, 'queue_time': 0.159431065, 'total_time': 0.158022154}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e6500-c7f4-77c2-ac78-09cc5224812c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 568, 'outpu

In [70]:
graph.invoke({"messages": [{"role": "user", "content": "I am doing B. Tech"}]}, t3)

[{'role': 'system', 'content': 'Conversation summary:\nHere\'s a summary of our conversation:\n\n* You introduced yourself as Aryan and mentioned that you are a Software Engineer (SWE).\n* We discussed your background and interests in software engineering.\n* You mentioned that you live in Noida, a major tech hub in India.\n* You\'re pursuing a B. Tech degree, likely in Computer Science and Engineering (CSE).\n* We had a few conversations where you repeated your name, "Aryan", multiple times, and I tried to respond accordingly.'}, HumanMessage(content='Hi, my name is Aryan', additional_kwargs={}, response_metadata={}, id='ea5eed27-0f65-4d2f-beb7-b1c6e0fb9c6a'), AIMessage(content="Aryan, it seems like you're repeating your name again. If you're ready, I can try to recall our previous conversation and we can continue from there. If not, we can start a new conversation. What would you prefer?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_token

{'messages': [HumanMessage(content='I am doing B. Tech', additional_kwargs={}, response_metadata={}, id='17cf8508-c021-4938-8101-273bb98e9a86'),
  AIMessage(content="So, you're pursuing a B. Tech degree, likely in Computer Science and Engineering (CSE) given your background as a Software Engineer. How's your experience been so far with your B. Tech program? Have you had any notable projects or internships that you've enjoyed working on?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 328, 'total_tokens': 388, 'completion_time': 0.17298539, 'completion_tokens_details': None, 'prompt_time': 0.089447787, 'prompt_tokens_details': None, 'queue_time': 0.160878691, 'total_time': 0.262433177}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e6501-2325-7080-a75e-8bd9e5da8a9a-0', tool_calls=[], invalid_